In [ ]:
#!/usr/bin/env python3
"""A basic console UNO game: you vs. AI.

Run:  python uno.py            (you + 2 computer players)
      python uno.py 3          (you + 3 computer players)
      python uno.py --bots     (watch computers play each other)
"""
import random
import sys
from collections import Counter

COLORS = ["Red", "Yellow", "Green", "Blue"]
ACTIONS = ["Skip", "Reverse", "Draw2"]
ANSI = {"Red": "\033[91m", "Yellow": "\033[93m", "Green": "\033[92m",
        "Blue": "\033[94m", "Wild": "\033[95m"}
RESET = "\033[0m"


def build_deck():
    deck = []
    for color in COLORS:
        deck.append((color, "0"))
        for value in [str(n) for n in range(1, 10)] + ACTIONS:
            deck += [(color, value)] * 2
    deck += [("Wild", "Wild")] * 4
    deck += [("Wild", "Wild4")] * 4
    random.shuffle(deck)
    return deck


def fmt(card):
    color, value = card
    label = value if color == "Wild" else f"{color} {value}"
    return f"{ANSI[color]}[{label}]{RESET}"


class Player:
    def __init__(self, name, human=False):
        self.name = name
        self.human = human
        self.hand = []


class UnoGame:
    def __init__(self, num_computers=2, all_bots=False):
        self.deck = build_deck()
        self.discard = []
        self.players = []
        if all_bots:
            self.players = [Player(f"CPU {i + 1}") for i in range(num_computers + 1)]
        else:
            self.players = [Player("You", human=True)]
            self.players += [Player(f"CPU {i + 1}") for i in range(num_computers)]
        for p in self.players:
            p.hand = [self.draw_card() for _ in range(7)]

        # Start with a plain number card on the discard pile
        first = self.draw_card()
        while first[1] in ACTIONS or first[0] == "Wild":
            self.deck.insert(0, first)
            random.shuffle(self.deck)
            first = self.draw_card()
        self.discard.append(first)
        self.current_color = first[0]
        self.turn = 0
        self.direction = 1

    # ---------- helpers ----------
    def draw_card(self):
        if not self.deck:  # reshuffle discard pile (keep top card)
            top = self.discard.pop()
            self.deck, self.discard = self.discard, [top]
            random.shuffle(self.deck)
        return self.deck.pop() if self.deck else None

    def draw_many(self, player, n):
        for _ in range(n):
            card = self.draw_card()
            if card:
                player.hand.append(card)

    def top(self):
        return self.discard[-1]

    def is_playable(self, card):
        color, value = card
        top_color, top_value = self.top()
        return color == "Wild" or color == self.current_color or value == top_value

    def next_index(self, steps=1):
        return (self.turn + self.direction * steps) % len(self.players)

    # ---------- choosing cards / colors ----------
    def choose_color_human(self):
        while True:
            pick = input("Choose a color (R/Y/G/B): ").strip().upper()
            for c in COLORS:
                if pick == c[0]:
                    return c
            print("Invalid color.")

    def choose_color_cpu(self, player):
        counts = Counter(c for c, _ in player.hand if c != "Wild")
        return counts.most_common(1)[0][0] if counts else random.choice(COLORS)

    def human_choose(self, player):
        playable = [c for c in player.hand if self.is_playable(c)]
        print("\nYour hand:")
        for i, card in enumerate(player.hand, 1):
            mark = "*" if self.is_playable(card) else " "
            print(f"  {i:2d}.{mark} {fmt(card)}")
        while True:
            prompt = "Pick a card number to play (* = playable), or 'd' to draw: "
            choice = input(prompt).strip().lower()
            if choice == "d":
                return None
            if choice.isdigit() and 1 <= int(choice) <= len(player.hand):
                card = player.hand[int(choice) - 1]
                if self.is_playable(card):
                    return card
                print("That card can't be played.")
            else:
                print("Invalid input.")
        return playable

    def cpu_choose(self, player):
        playable = [c for c in player.hand if self.is_playable(c)]
        if not playable:
            return None
        non_wild = [c for c in playable if c[0] != "Wild"]
        actions = [c for c in non_wild if c[1] in ACTIONS]
        return random.choice(actions or non_wild or playable)

    # ---------- playing ----------
    def play_card(self, player, card):
        player.hand.remove(card)
        self.discard.append(card)
        color, value = card
        if color == "Wild":
            self.current_color = (self.choose_color_human() if player.human
                                  else self.choose_color_cpu(player))
        else:
            self.current_color = color
        print(f"{player.name} played {fmt(card)}"
              + (f" and chose {self.current_color}" if color == "Wild" else ""))

        # Card effects
        if value == "Skip":
            victim = self.players[self.next_index()]
            print(f"{victim.name} is skipped!")
            self.turn = self.next_index()
        elif value == "Reverse":
            self.direction *= -1
            print("Direction reversed!")
            if len(self.players) == 2:  # acts like Skip with two players
                self.turn = self.next_index()
        elif value == "Draw2":
            victim = self.players[self.next_index()]
            self.draw_many(victim, 2)
            print(f"{victim.name} draws 2 cards and is skipped!")
            self.turn = self.next_index()
        elif value == "Wild4":
            victim = self.players[self.next_index()]
            self.draw_many(victim, 4)
            print(f"{victim.name} draws 4 cards and is skipped!")
            self.turn = self.next_index()

    def take_turn(self):
        player = self.players[self.turn]
        print("\n" + "=" * 50)
        print(f"{player.name}'s turn | Top card: {fmt(self.top())} | "
              f"Color: {ANSI[self.current_color]}{self.current_color}{RESET}")
        print("Cards: " + ", ".join(f"{p.name}={len(p.hand)}" for p in self.players))

        card = self.human_choose(player) if player.human else self.cpu_choose(player)

        if card is None:  # draw a card
            drawn = self.draw_card()
            if drawn is None:
                print(f"{player.name} can't draw (deck empty).")
            else:
                player.hand.append(drawn)
                if player.human:
                    print(f"You drew {fmt(drawn)}")
                else:
                    print(f"{player.name} draws a card.")
                if self.is_playable(drawn):
                    if player.human:
                        again = input("Play it? (y/n): ").strip().lower()
                        if again == "y":
                            card = drawn
                    else:
                        card = drawn
        if card is not None:
            self.play_card(player, card)
            if len(player.hand) == 1:
                print(f"*** {player.name} says UNO! ***")
            if not player.hand:
                return True
        return False

    def run(self):
        print("Welcome to UNO!")
        while True:
            if self.take_turn():
                winner = self.players[self.turn]
                print("\n" + "=" * 50)
                print(f"🎉 {winner.name} wins the game!")
                break
            self.turn = self.next_index()


if __name__ == "__main__":
    args = sys.argv[1:]
    bots_only = "--bots" in args
    nums = [a for a in args if a.isdigit()]
    cpus = int(nums[0]) if nums else 2
    cpus = max(1, min(cpus, 5))
    UnoGame(cpus, all_bots=bots_only).run()

Welcome to UNO!

You's turn | Top card: [Green 9] | Color: Green
Cards: You=7, CPU 1=7, CPU 2=7

Your hand:
   1.* [Wild]
   2.* [Green Draw2]
   3.  [Yellow Draw2]
   4.* [Green 7]
   5.  [Blue Reverse]
   6.* [Green Skip]
   7.  [Red 8]


Pick a card number to play (* = playable), or 'd' to draw:  4


You played [Green 7]

CPU 1's turn | Top card: [Green 7] | Color: Green
Cards: You=6, CPU 1=7, CPU 2=7
CPU 1 played [Wild] and chose Red

CPU 2's turn | Top card: [Wild] | Color: Red
Cards: You=6, CPU 1=6, CPU 2=7
CPU 2 played [Red Reverse]
Direction reversed!

CPU 1's turn | Top card: [Red Reverse] | Color: Red
Cards: You=6, CPU 1=6, CPU 2=6
CPU 1 played [Red Reverse]
Direction reversed!

CPU 2's turn | Top card: [Red Reverse] | Color: Red
Cards: You=6, CPU 1=5, CPU 2=6
CPU 2 played [Green Reverse]
Direction reversed!

CPU 1's turn | Top card: [Green Reverse] | Color: Green
Cards: You=6, CPU 1=5, CPU 2=5
CPU 1 draws a card.
CPU 1 played [Yellow Reverse]
Direction reversed!

CPU 2's turn | Top card: [Yellow Reverse] | Color: Yellow
Cards: You=6, CPU 1=5, CPU 2=5
CPU 2 played [Wild] and chose Blue

You's turn | Top card: [Wild] | Color: Blue
Cards: You=6, CPU 1=5, CPU 2=4

Your hand:
   1.* [Wild]
   2.  [Green Draw2]
   3.  [Yellow Draw2]
   4.* [Blue Reverse]
   5.  [Green Skip]
   6.

Pick a card number to play (* = playable), or 'd' to draw:  d


You drew [Red 5]

CPU 1's turn | Top card: [Wild] | Color: Blue
Cards: You=7, CPU 1=5, CPU 2=4
CPU 1 played [Blue 0]

CPU 2's turn | Top card: [Blue 0] | Color: Blue
Cards: You=7, CPU 1=4, CPU 2=4
CPU 2 played [Blue Draw2]
You draws 2 cards and is skipped!

CPU 1's turn | Top card: [Blue Draw2] | Color: Blue
Cards: You=9, CPU 1=4, CPU 2=3
CPU 1 draws a card.

CPU 2's turn | Top card: [Blue Draw2] | Color: Blue
Cards: You=9, CPU 1=5, CPU 2=3
CPU 2 played [Blue 1]

You's turn | Top card: [Blue 1] | Color: Blue
Cards: You=9, CPU 1=5, CPU 2=2

Your hand:
   1.* [Wild]
   2.  [Green Draw2]
   3.  [Yellow Draw2]
   4.* [Blue Reverse]
   5.  [Green Skip]
   6.  [Red 8]
   7.  [Red 5]
   8.* [Blue 7]
   9.  [Yellow 2]


Pick a card number to play (* = playable), or 'd' to draw:  8


You played [Blue 7]

CPU 1's turn | Top card: [Blue 7] | Color: Blue
Cards: You=8, CPU 1=5, CPU 2=2
CPU 1 played [Green 7]

CPU 2's turn | Top card: [Green 7] | Color: Green
Cards: You=8, CPU 1=4, CPU 2=2
CPU 2 played [Green 9]
*** CPU 2 says UNO! ***

You's turn | Top card: [Green 9] | Color: Green
Cards: You=8, CPU 1=4, CPU 2=1

Your hand:
   1.* [Wild]
   2.* [Green Draw2]
   3.  [Yellow Draw2]
   4.  [Blue Reverse]
   5.* [Green Skip]
   6.  [Red 8]
   7.  [Red 5]
   8.  [Yellow 2]


Pick a card number to play (* = playable), or 'd' to draw:  1
Choose a color (R/Y/G/B):  R


You played [Wild] and chose Red

CPU 1's turn | Top card: [Wild] | Color: Red
Cards: You=7, CPU 1=4, CPU 2=1
CPU 1 played [Red 6]

CPU 2's turn | Top card: [Red 6] | Color: Red
Cards: You=7, CPU 1=3, CPU 2=1
CPU 2 played [Wild4] and chose Yellow
You draws 4 cards and is skipped!

🎉 You wins the game!
